# Data prep

Turns the raw CSVs into model-ready artifacts, encoding what `eda.ipynb` established. Every choice below is
either a structural fact from the EDA or a decision made by measurement in the CV harness — nothing here is
convention.

**Design rule: anything fitted must be fittable inside a fold.** So this notebook does *not* impute, select
features or transform the target and then save the result — those all leak. It saves the cleaned data with
NaNs intact plus a manifest, and the fold-safe transforms live in `src/qrt_prep.py` for the modelling
notebook to import.

What actually gets decided here:

| step | decision | why |
|---|---|---|
| redundant columns | drop 3 | exact sign-flips, singular design matrix |
| `DAY_ID` | grouping key, never a feature | shuffled, no target signal |
| country | split every model | FR/DE rows are bit-identical, only `COUNTRY` differs |
| imputation | fold median | measured; all options within noise |
| target | rank transform | measured, +0.04 — the biggest single win |
| feature selection | permutation null, **inside the fold** | full-train selection inflates CV by ~0.008 |

Reference score at the end: **≈0.280** pooled OOF Spearman, honestly measured.

In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import RidgeCV

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
sys.path.insert(0, str(ROOT / "src"))
import qrt_prep as P

sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 200)
RIDGE = lambda: RidgeCV(alphas=np.logspace(-2, 4, 40))
print(f"repo root: {ROOT}")

repo root: /Users/ryan/QRT-Electricity-Forecasting


## 1. Load and validate

`build_frames` merges X/y, drops the three sign-flip duplicates, and leaves NaNs alone.

`validate` re-asserts every structural invariant the EDA depends on. It's cheap and it turns those findings
into a regression test — if the organisers ever reissue the data, or a merge goes wrong, this fails loudly
instead of silently changing what the models mean.

In [2]:
train, test = P.build_frames(ROOT / "data" / "raw")
FEATURES = P.feature_columns(train)

for name, ok in P.validate(train, test).items():
    print(f"  [{'ok' if ok else 'FAIL'}]  {name}")
print(f"\ntrain {train.shape}   test {test.shape}   features {len(FEATURES)}")
print(f"dropped as exact sign-flips: {P.REDUNDANT}")

  [ok]  no shared days
  [ok]  no shared IDs
  [ok]  same features
  [ok]  no DE-only days
  [ok]  train: FR/DE rows identical
  [ok]  test: FR/DE rows identical
  [ok]  redundant cols dropped

train (1494, 33)   test (654, 32)   features 29
dropped as exact sign-flips: ['DE_NET_IMPORT', 'FR_NET_IMPORT', 'FR_DE_EXCHANGE']


The `FR/DE rows identical` check is the important one. It is the fact that forces per-country modelling, and
it is the reason the official benchmark caps out where it does — a model without `COUNTRY` must emit one
number for two rows that have different targets.

## 2. Column manifest

`DAY_ID` stays in the frame but never enters a design matrix: it's shuffled, carries no target signal, and
its only job is grouping the CV folds so a day's two rows can't straddle a split.

In [3]:
for a, b in P.SIGN_FLIP_PAIRS:                    # confirm the drop was justified
    raw = pd.read_csv(ROOT / "data/raw/X_train.csv")
    print(f"  {a:15s} + {b:15s}  sum sd {(raw[a] + raw[b]).std():.1e}")

GROUPS = {
    "commodity": ["GAS_RET", "COAL_RET", "CARBON_RET"],
    "weather":   [c for c in FEATURES if c.split("_", 1)[1] in ("TEMP", "RAIN", "WIND")],
    "generation":[c for c in FEATURES if c.split("_", 1)[1] in
                  ("GAS", "COAL", "HYDRO", "NUCLEAR", "SOLAR", "WINDPOW", "LIGNITE")],
    "load":      [c for c in FEATURES if c.split("_", 1)[1] in
                  ("CONSUMPTION", "RESIDUAL_LOAD", "NET_EXPORT", "NET_IMPORT")],
    "exchange":  [c for c in FEATURES if "EXCHANGE" in c],
}
print()
for k, v in GROUPS.items():
    print(f"  {k:<11}{len(v):>3}  {v[:4]}{' ...' if len(v) > 4 else ''}")
assert sum(len(v) for v in GROUPS.values()) == len(FEATURES), "manifest does not cover every feature"
print(f"\ncovered {sum(len(v) for v in GROUPS.values())}/{len(FEATURES)} features")

  DE_NET_EXPORT   + DE_NET_IMPORT    sum sd 0.0e+00
  FR_NET_EXPORT   + FR_NET_IMPORT    sum sd 0.0e+00
  DE_FR_EXCHANGE  + FR_DE_EXCHANGE   sum sd 0.0e+00

  commodity    3  ['GAS_RET', 'COAL_RET', 'CARBON_RET']
  weather      6  ['DE_RAIN', 'FR_RAIN', 'DE_WIND', 'FR_WIND'] ...
  generation  13  ['DE_GAS', 'FR_GAS', 'DE_COAL', 'FR_COAL'] ...
  load         6  ['DE_CONSUMPTION', 'FR_CONSUMPTION', 'DE_NET_EXPORT', 'FR_NET_EXPORT'] ...
  exchange     1  ['DE_FR_EXCHANGE']

covered 29/29 features


## 3. CV harness

Group on `DAY_ID`, and score by pooling every out-of-fold prediction into a **single** Spearman — that's what
the leaderboard computes. Averaging per-fold scores would hide cross-country miscalibration.

One trap worth encoding: `GroupKFold` is deterministic. It bins groups by size and ignores row order, so
repeating it returns an identical partition and `std = 0.0000` — which looks like a stable result and is
actually no measurement at all. `make_folds` randomises the group→fold assignment so the ± is real.

In [4]:
f0 = P.make_folds(train.DAY_ID, seed=0)
f1 = P.make_folds(train.DAY_ID, seed=1)
print(f"folds: {len(f0)}   sizes {[len(v) for _, v in f0]}")
print(f"seed 0 vs seed 1 identical? {all(np.array_equal(a[1], b[1]) for a, b in zip(f0, f1))}")

# a day's two rows must always land in the same fold
leaks = sum(len(set(train.iloc[tr].DAY_ID) & set(train.iloc[va].DAY_ID)) for tr, va in f0)
print(f"days straddling a train/val split: {leaks}")

def reference(cols_fr, cols_de=None, target=P.rank_transform, impute="median", select=None, seeds=range(10)):
    cols_de = cols_de or cols_fr
    def fit_predict(t, v, country):
        cols = select(t, FEATURES, country) if select else (cols_fr if country == "FR" else cols_de)
        A, B = P.impute(t, v, cols=cols, strategy=impute)
        return RIDGE().fit(A, target(t.TARGET.values)).predict(B)
    return P.cross_validate(train, fit_predict, seeds=seeds)

folds: 5   sizes [299, 294, 291, 306, 304]
seed 0 vs seed 1 identical? False
days straddling a train/val split: 0


## 4. Decisions by measurement

### 4.1 Imputation

In [5]:
for s in ["zero", "mean", "median"]:
    m, sd, _ = reference(FEATURES, impute=s)
    print(f"  {s:<8}{m:+.4f} +/- {sd:.4f}")

  zero    +0.2757 +/- 0.0078


  mean    +0.2758 +/- 0.0079


  median  +0.2759 +/- 0.0078


Indistinguishable. Missingness is day-blocked, matched between train and test (6.3% vs 6.1%), and carries no
target signal, so there's nothing to recover. Going with **fold median** — fold-safe and robust, and the
choice is free.

The EDA also showed adding missing-indicator columns *hurts*, which is the first instance of a pattern that
holds throughout: this dataset punishes added capacity.

### 4.2 Target transform

In [6]:
for nm, tf in [("raw", lambda v: v),
               ("winsorised 5%", lambda v: np.clip(v, *np.quantile(v, [.05, .95]))),
               ("signed sqrt", lambda v: np.sign(v) * np.sqrt(np.abs(v))),
               ("rank", P.rank_transform)]:
    m, sd, _ = reference(FEATURES, target=tf)
    print(f"  {nm:<16}{m:+.4f} +/- {sd:.4f}")

  raw             +0.2357 +/- 0.0096


  winsorised 5%   +0.2577 +/- 0.0086


  signed sqrt     +0.2714 +/- 0.0076


  rank            +0.2759 +/- 0.0078


**Rank wins by ~0.04, five-plus sigma.** Straight out of the EDA: 6–8 days carry ~31% of the squared-error
weight but ~1% of the rank positions, so least squares spends most of its effort where Spearman doesn't
look. Winsorising and signed-sqrt recover most of the gain, which confirms the mechanism is tail suppression
rather than anything ranks-specific.

Free move — the metric only sees ranks, so any monotone transform of the target costs nothing.

### 4.3 Feature selection, and where the EDA was optimistic

In [7]:
select_infold = lambda t, cols: P.select_by_permutation_null(t, cols, seed=0, n_perm=1000)
FULL_FR = P.select_by_permutation_null(train[train.COUNTRY == "FR"], FEATURES, seed=0)
FULL_DE = P.select_by_permutation_null(train[train.COUNTRY == "DE"], FEATURES, seed=0)
print(f"selected on the full training set -- FR {FULL_FR}\n{'':34}DE ({len(FULL_DE)}) {FULL_DE}\n")

m0, s0, _ = reference(FEATURES)
m1, s1, _ = reference(FULL_FR, FEATURES)
# select for FR only; DE has enough signal to use the full set (eda.ipynb 6.3)
fr_only = lambda t, c, country: select_infold(t, c) if country == "FR" else c
m2, s2, _ = reference(FEATURES, select=fr_only)
print(f"  {'FR: all 29 features':<34}{m0:+.4f} +/- {s0:.4f}")
print(f"  {'FR: selected on full train':<34}{m1:+.4f} +/- {s1:.4f}   <- optimistic")
print(f"  {'FR: selected inside each fold':<34}{m2:+.4f} +/- {s2:.4f}   <- honest")

selected on the full training set -- FR ['FR_WINDPOW', 'GAS_RET', 'CARBON_RET']
                                  DE (9) ['DE_NET_EXPORT', 'DE_GAS', 'DE_COAL', 'DE_HYDRO', 'DE_WINDPOW', 'FR_WINDPOW', 'DE_LIGNITE', 'DE_RESIDUAL_LOAD', 'DE_WIND']



  FR: all 29 features               +0.2759 +/- 0.0078
  FR: selected on full train        +0.2886 +/- 0.0072   <- optimistic
  FR: selected inside each fold     +0.2806 +/- 0.0066   <- honest


This is the one place the EDA overstated a result, and it's worth being explicit about.

`eda.ipynb` §6.3 selected France's features once on the full training set and then reported CV on the
result — so every fold's "held-out" rows had already voted on which columns to use. The honest fold-internal
version gives back about **0.008** of that gain.

Two things follow. The CV *estimate* of 0.2896 was inflated and the realistic figure is ~0.280 — that's the
number to carry forward. But the *model* is fine: selecting on all training data and then predicting the
real test set is entirely legitimate, since the test labels were never involved. Only the self-assessment
was contaminated.

The selection is stable regardless — the same three French features come back in most folds, which is why
the leak was small rather than catastrophic.

In [8]:
picks = {}
for seed in range(4):
    for i, (tr, _) in enumerate(P.make_folds(train.DAY_ID, seed=seed)):
        t = train.iloc[tr]
        s = tuple(sorted(select_infold(t[t.COUNTRY == "FR"], FEATURES)))
        picks[s] = picks.get(s, 0) + 1
print("FR features chosen across 20 folds:")
for s, n in sorted(picks.items(), key=lambda kv: -kv[1]):
    print(f"  {n:>3}x  {list(s)}")

FR features chosen across 20 folds:
   11x  ['CARBON_RET', 'FR_WINDPOW', 'GAS_RET']
    5x  ['CARBON_RET', 'GAS_RET']
    2x  ['CARBON_RET', 'DE_HYDRO', 'FR_WINDPOW', 'GAS_RET']
    1x  ['CARBON_RET', 'FR_COAL', 'GAS_RET']
    1x  ['CARBON_RET']


### 4.4 Standard prep steps that don't apply here

Row deletion, normalisation and centering are the reflex next steps. All three are measured below, and none
of them earn a place. Recording the numbers so nobody re-litigates it later.

Note the hard constraint on row deletion: **every one of the 654 test rows needs a prediction**, so it can
only ever be applied to training data — which means train and test would be preprocessed differently.

In [9]:
from scipy.stats import norm

def _std(A, B):                          # z-score using TRAIN-fold statistics only
    mu, sd = A.mean(), A.std().replace(0, 1)
    return (A - mu) / sd, (B - mu) / sd

def _center(A, B):
    mu = A.mean(); return A - mu, B - mu

def _winsor(A, B):
    lo, hi = A.quantile(.01), A.quantile(.99)
    return A.clip(lo, hi, axis=1), B.clip(lo, hi, axis=1)

def _rank_feats(A, B):                   # each feature -> its quantile in the train fold
    f = lambda D: pd.DataFrame({c: np.searchsorted(np.sort(A[c].values), D[c].values) / len(A)
                                for c in A.columns}, index=D.index)
    return f(A), f(B)

def _gauss_rank(A, B):
    a, b = _rank_feats(A, B)
    f = lambda X: pd.DataFrame(norm.ppf(np.clip(X.values, 1e-3, 1 - 1e-3)),
                               columns=X.columns, index=X.index)
    return f(a), f(b)

def variant(prep=None, rows=None, seeds=range(10)):
    def fit_predict(t, v, country):
        if rows is not None: t = rows(t)
        A, B = P.impute(t, v, cols=FEATURES, strategy="median")
        if prep is not None: A, B = prep(A, B)
        return RIDGE().fit(A, P.rank_transform(t.TARGET.values)).predict(B)
    return P.cross_validate(train, fit_predict, seeds=seeds)

print(f"{'baseline (no feature prep)':<30}{variant()[0]:+.4f}")
for nm, fn in [("center only", _center), ("standardise (z-score)", _std),
               ("winsorise features 1%", _winsor), ("rank -> uniform", _rank_feats),
               ("rank -> gaussian", _gauss_rank)]:
    m, sd, _ = variant(fn); print(f"  {nm:<28}{m:+.4f} +/- {sd:.4f}")

anyna = lambda t: t[~t[FEATURES].isna().any(axis=1)]
m, sd, _ = variant(rows=anyna)
lost = train[FEATURES].isna().any(axis=1)
print(f"\n  {'complete-case training':<28}{m:+.4f} +/- {sd:.4f}"
      f"   (drops {lost.sum()} rows, {lost.mean()*100:.0f}% of train)")
print(f"  {"test rows needing prediction":<28}  {len(test)} -- deletion is not available there")

baseline (no feature prep)    +0.2759


  center only                 +0.2759 +/- 0.0078


  standardise (z-score)       +0.2757 +/- 0.0078


  winsorise features 1%       +0.2779 +/- 0.0081


  rank -> uniform             +0.2679 +/- 0.0077


  rank -> gaussian            +0.2756 +/- 0.0092



  complete-case training      +0.2722 +/- 0.0088   (drops 218 rows, 15% of train)
  test rows needing prediction  654 -- deletion is not available there


**Centering is exactly a no-op** — identical to four decimals, because ridge fits an intercept that already
absorbs the mean, and sklearn does not penalise that intercept.

**Standardising changes nothing** (0.2757 vs 0.2759). The columns arrive pre-standardised by the organisers,
and the residual within-country spread is only ~2.5×, which `RidgeCV`'s alpha search over six orders of
magnitude absorbs without noticing.

**Deleting incomplete rows loses.** It costs 218 training rows (15%), breaks 47 day-pairings, and scores
worse. Missingness here is day-blocked, matched between train and test, and carries no target signal — there
is nothing to remove, only data to lose.

**Rank-transforming features hurts** (0.268), which is worth contrasting with the target, where the same
transform was the single biggest win. The asymmetry is real: the *target's* tails distort the loss, so
flattening them helps; the *features'* tails carry genuine information — an extreme wind day really is
extreme — and ranking discards the magnitude that made it informative.

Winsorising features at 1% is the only variant pointing upward (+0.002), and it sits well inside the ±0.008
noise band. Not banking on it, though it is at least consistent with the tail story.

One forward-looking exception. None of this changes the *ridge* solution, but the JAX models coming next are
trained by gradient descent, where conditioning affects convergence rather than the optimum. Standardising
takes the condition number of XᵀX from ~30,100 to ~23,500 and reaches a lower loss for the same step budget.
Small, but free — so standardise inside the fold for the gradient-based models even though it is pointless
for ridge.

## 5. Build and save

In [10]:
manifest = {
    "source": "data/raw",
    "id_cols": P.ID_COLS,
    "features": FEATURES,
    "dropped_redundant": P.REDUNDANT,
    "feature_groups": GROUPS,
    "group_key": "DAY_ID",
    "countries": P.COUNTRIES,
    "policy": {"impute": "median, fitted per fold",
               "target": "rank_transform, fitted per fold",
               "selection": "permutation null (5% FWER), fitted per fold"},
    "selected_full_train": {"FR": FULL_FR, "DE": FULL_DE},
    "reference_cv": {"metric": "pooled OOF Spearman, 10 randomised group-fold seeds",
                     "all_features": round(m0, 4), "in_fold_selection": round(m2, 4)},
    "n_train": len(train), "n_test": len(test),
}
out = P.save(train, test, manifest, ROOT / "data" / "processed")
for f in sorted(out.iterdir()):
    print(f"  {f.name:<16}{f.stat().st_size/1024:8.1f} KB")

  manifest.json        2.1 KB
  test.parquet       126.5 KB
  train.parquet      287.8 KB


## 6. Verify from the saved artifacts

Reload from disk and confirm the round-trip is lossless and the reference score reproduces. Cheap insurance
against a prep bug quietly poisoning every downstream experiment.

In [11]:
tr2, te2, man = P.load_processed(ROOT / "data" / "processed")
print(f"round-trip equal: train {tr2.equals(train)}   test {te2.equals(test)}")
print(f"NaNs preserved:   train {int(tr2[FEATURES].isna().sum().sum())} "
      f"  test {int(te2[FEATURES].isna().sum().sum())}")
print(f"manifest features match: {man['features'] == FEATURES}")

m, sd, oof = reference(FEATURES, select=fr_only, seeds=range(5))
print(f"\nreference model from artifacts: {m:+.4f} +/- {sd:.4f}")

fr = (train.COUNTRY == "FR").values
print(f"  within-country  FR {P.pooled_spearman(oof[fr], train.TARGET[fr]):+.4f}"
      f"   DE {P.pooled_spearman(oof[~fr], train.TARGET[~fr]):+.4f}")

round-trip equal: train True   test True
NaNs preserved:   train 783   test 320
manifest features match: True



reference model from artifacts: +0.2818 +/- 0.0086
  within-country  FR +0.1968   DE +0.3295


## 7. Submission path

Final fit uses all training rows — selection on the full training set is legitimate here because the test
labels are never touched. Writing a reference submission so the plumbing is proven end to end before any
modelling work starts.

In [12]:
pred = np.zeros(len(test))
for c in P.COUNTRIES:
    t = train[train.COUNTRY == c]
    cols = FULL_FR if c == "FR" else FEATURES
    mask = (test.COUNTRY == c).values
    A, B = P.impute(t, test[mask], cols=cols, strategy="median")
    pred[mask] = RIDGE().fit(A, P.rank_transform(t.TARGET.values)).predict(B)

sub = P.make_submission(test, pred, ROOT / "submissions" / "ridge_rank_reference.csv")
print(sub.describe().T.to_string())
print(f"\nrows {len(sub)}  |  IDs match test: {(sub.ID.values == test.ID.values).all()}")
print(sub.head())

        count         mean         std       min         25%          50%          75%          max
ID      654.0  1075.192661  625.699109  1.000000  528.500000  1060.500000  1631.500000  2147.000000
TARGET  654.0     0.507301    0.090449  0.166375    0.461531     0.506231     0.560047     0.838855

rows 654  |  IDs match test: True
     ID    TARGET
0  1115  0.498965
1  1202  0.547961
2  1194  0.441992
3  1084  0.537867
4  1135  0.457127


Predictions land on the rank scale (0, 1] rather than the target's, which is fine — Spearman only reads
ordering. Worth remembering when eyeballing a submission: these are not price changes.

## 8. Summary

Artifacts in `data/processed/`: `train.parquet`, `test.parquet`, `manifest.json`. Shared code in
`src/qrt_prep.py`.

Settled here:

- 29 features after dropping three exact sign-flips; `DAY_ID` is a grouping key, not a feature.
- Per-country models, always — FR and DE rows are bit-identical apart from `COUNTRY`.
- Fold median imputation (free choice), rank-transformed target (+0.04, the main win), permutation-null
  feature selection fitted inside the fold.
- Reference: **≈0.280** pooled OOF Spearman. The EDA's 0.2896 carried ~0.008 of selection leakage.

Deliberately not done, all measured rather than assumed:

- from `eda.ipynb` §6.4 — FR−DE spreads, seasonal phase terms, missing-value indicators, gradient boosting,
  volatility scaling;
- from §4.4 above — row deletion, centering, standardisation, feature rank transforms.

The organisers already standardised the columns and blocked the missingness cleanly, so most of the routine
prep checklist has nothing left to do. What actually moved the score was structural (split by country, group
the folds) and metric-aligned (rank the target). The prep stays minimal on purpose.

Next: `vmap` the fold × country × alpha grid, and replace the rank transform with a differentiable soft-rank
objective so the model optimises Spearman directly instead of approximating it through the target.